# GP posterior correlation vs. number of context points

1D GP prior on a 256-point grid over `[0, 1]` with an RBF kernel. We pick a
growing set of context points (first `n_context` points of a fixed random
permutation, so increasing the slider only *adds* points), observe a fixed
ground-truth draw `f_true` at those locations, and compute the GP posterior.

Left panel: posterior mean +/- 2 std, a few posterior sample draws, `f_true`,
and the context observations. Right panel: the posterior **correlation**
matrix (normalized posterior covariance) over the grid, which should show
the covariance collapsing to a spike pattern around context points as
`n_context` grows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import IntSlider
from IPython.display import display, clear_output

%matplotlib inline

rng = np.random.default_rng(42)

n = 256
x = np.linspace(0, 1, n)
length_scale = 0.3
noise_var = 1e-6  # small nugget: keeps K_CC well-conditioned, context ~noise-free


def rbf_1d(x, length_scale=length_scale):
    diff = x[:, None] - x[None, :]
    return np.exp(-0.5 * diff ** 2 / length_scale ** 2)


K = rbf_1d(x) + 1e-9 * np.eye(n)
L = np.linalg.cholesky(K)

# One fixed ground-truth draw, and a fixed random order in which points
# become "observed" as n_context grows.
f_true = L @ rng.standard_normal(n)
context_order = rng.permutation(n)

In [ ]:
def gp_posterior(n_context, n_samples=3):
    """Condition the prior on the first n_context points of context_order."""
    if n_context == 0:
        post_mean = np.zeros(n)
        post_cov = K.copy()
        ctx_idx = np.array([], dtype=int)
    else:
        ctx_idx = context_order[:n_context]
        y_c = f_true[ctx_idx]

        K_cc = K[np.ix_(ctx_idx, ctx_idx)] + noise_var * np.eye(n_context)
        K_xc = K[:, ctx_idx]

        L_cc = np.linalg.cholesky(K_cc)
        alpha = np.linalg.solve(L_cc.T, np.linalg.solve(L_cc, y_c))
        v = np.linalg.solve(L_cc, K_xc.T)

        post_mean = K_xc @ alpha
        post_cov = K - v.T @ v

    # numerical floor so the correlation normalization doesn't blow up
    diag = np.clip(np.diag(post_cov), 1e-10, None)
    std = np.sqrt(diag)
    post_corr = post_cov / np.outer(std, std)

    samples = None
    if n_samples > 0:
        cov_jittered = post_cov + 1e-9 * np.eye(n)
        L_post = np.linalg.cholesky(cov_jittered)
        samples = post_mean[:, None] + L_post @ rng.standard_normal((n, n_samples))

    return post_mean, post_cov, post_corr, ctx_idx, samples

In [ ]:
output = widgets.Output()
slider = IntSlider(min=0, max=n, step=1, value=5, description="n_context")


def plot_posterior(n_context):
    post_mean, post_cov, post_corr, ctx_idx, samples = gp_posterior(n_context)
    std = np.sqrt(np.clip(np.diag(post_cov), 0, None))

    fig, (ax_gp, ax_corr) = plt.subplots(1, 2, figsize=(12, 5))

    ax_gp.plot(x, f_true, color="k", lw=1, ls="--", label="f_true")
    ax_gp.plot(x, post_mean, color="C0", lw=2, label="posterior mean")
    ax_gp.fill_between(
        x, post_mean - 2 * std, post_mean + 2 * std,
        color="C0", alpha=0.2, label="+/- 2 std",
    )
    for i in range(samples.shape[1]):
        ax_gp.plot(x, samples[:, i], color="C1", lw=0.8, alpha=0.6)
    if n_context > 0:
        ax_gp.scatter(
            x[ctx_idx], f_true[ctx_idx],
            color="red", zorder=5, label="context points",
        )
    ax_gp.set_title(f"Posterior GP (n_context={n_context})")
    ax_gp.set_xlabel("x")
    ax_gp.set_ylim(-3, 3)
    ax_gp.legend(loc="upper right", fontsize=8)

    im = ax_corr.imshow(
        post_corr, origin="lower", extent=[0, 1, 0, 1],
        cmap="RdBu_r", vmin=-1, vmax=1,
    )
    ax_corr.set_title("Posterior correlation matrix")
    ax_corr.set_xlabel("x")
    ax_corr.set_ylabel("x'")
    fig.colorbar(im, ax=ax_corr, fraction=0.046, pad=0.04)

    fig.tight_layout()
    plt.show()
    plt.close(fig)  # avoid accumulating open figures across slider moves

    # off-diagonal correlation stats: how strongly points still co-vary
    # once n_context observations have "explained away" the prior correlation
    off_diag = ~np.eye(n, dtype=bool)
    offdiag_vals = post_corr[off_diag]
    abs_offdiag = np.abs(offdiag_vals)

    stats = {
        "mean |off-diag corr|": abs_offdiag.mean(),
        "median |off-diag corr|": np.median(abs_offdiag),
        "max |off-diag corr|": abs_offdiag.max(),
        "std(off-diag corr)": offdiag_vals.std(),
        "mean posterior std": std.mean(),
        "max posterior std": std.max(),
        "RMSE(post_mean, f_true)": np.sqrt(np.mean((post_mean - f_true) ** 2)),
    }
    for name, val in stats.items():
        print(f"{name:26s} = {val:.4f}")


def on_slider_change(change):
    with output:
        clear_output(wait=True)
        plot_posterior(change["new"])


slider.observe(on_slider_change, names="value")
display(slider, output)
with output:
    plot_posterior(slider.value)